# Assignment: Predicting Insurance Charges using Multiple Linear Regression

In [ ]:
# Question 1: Load the dataset, display first 10 rows, and check for missing values

import pandas as pd

df = pd.read_csv('insurance (1).csv')

print("First 10 Rows:")
display(df.head(10))

print("\nMissing Values:")
print(df.isnull().sum())


In [ ]:
# Question 2: Generate summary statistics (mean, median, std) for numeric features

numeric_cols = df.select_dtypes(include=['int64','float64']).columns

summary_stats = pd.DataFrame({
    'Mean': df[numeric_cols].mean(),
    'Median': df[numeric_cols].median(),
    'Std Dev': df[numeric_cols].std()
})

display(summary_stats)


In [ ]:
# Question 3: Encode categorical features using One-Hot Encoding

df_encoded = pd.get_dummies(df, columns=['sex','smoker','region'], drop_first=True)

print("Encoded Dataset Shape:", df_encoded.shape)
display(df_encoded.head())


In [ ]:
# Question 4: Create a Scatter Plot of BMI vs Charges

import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.scatter(df['bmi'], df['charges'])
plt.xlabel('BMI')
plt.ylabel('Charges')
plt.title('BMI vs Insurance Charges')
plt.show()


In [ ]:
# Question 5: Create a Box Plot of Charges by Smoker Status

import matplotlib.pyplot as plt

df.boxplot(column='charges', by='smoker', figsize=(7,5))
plt.title('Charges by Smoker Status')
plt.suptitle('')
plt.xlabel('Smoker')
plt.ylabel('Charges')
plt.show()


In [ ]:
# Question 6: Create Histograms of Age and Charges

import matplotlib.pyplot as plt

df[['age','charges']].hist(figsize=(10,4))
plt.tight_layout()
plt.show()


In [ ]:
# Question 7: Split the dataset into training (80%) and testing (20%) sets

from sklearn.model_selection import train_test_split

X = df_encoded.drop('charges', axis=1)
y = df_encoded['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)


In [ ]:
# Question 8: Build a Multiple Linear Regression Model and Predict Charges

from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("First 10 Predictions:")
print(y_pred[:10])


In [ ]:
# Question 9: Evaluate the Model using R-squared, MAE and MSE

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("R-Squared :", r2)
print("MAE       :", mae)
print("MSE       :", mse)


In [ ]:
# Question 10: Display Model Coefficients and Intercept

coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
})

display(coefficients.sort_values(by='Coefficient', ascending=False))

print("Intercept:", model.intercept_)

# Interpretation:
# Positive coefficient -> charges increase as feature increases.
# Negative coefficient -> charges decrease as feature increases.


In [ ]:
# Question 11: Check Multicollinearity using Correlation Matrix

import seaborn as sns
import matplotlib.pyplot as plt

corr_matrix = df_encoded.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


In [ ]:
# Question 12: Apply Feature Scaling and Re-evaluate Model

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42
)

scaled_model = LinearRegression()
scaled_model.fit(X_train_s, y_train_s)

y_pred_s = scaled_model.predict(X_test_s)

print("Scaled Model R2 :", r2_score(y_test_s, y_pred_s))
print("Scaled Model MAE:", mean_absolute_error(y_test_s, y_pred_s))
print("Scaled Model MSE:", mean_squared_error(y_test_s, y_pred_s))

# Note:
# Linear regression generally gives the same performance with or without scaling.


In [ ]:
# Question 13: Drop Less Important Features and Re-evaluate

# Example: Dropping region features

reduced_features = [col for col in X.columns if not col.startswith('region_')]

X_reduced = X[reduced_features]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reduced, y, test_size=0.20, random_state=42
)

reduced_model = LinearRegression()
reduced_model.fit(X_train_r, y_train_r)

y_pred_r = reduced_model.predict(X_test_r)

print("Reduced Model R2 :", r2_score(y_test_r, y_pred_r))
print("Reduced Model MAE:", mean_absolute_error(y_test_r, y_pred_r))
print("Reduced Model MSE:", mean_squared_error(y_test_r, y_pred_r))


In [ ]:
# Question 14: Which features have the strongest effect on insurance charges? Explain.

abs_coef = coefficients.copy()
abs_coef['Absolute_Coefficient'] = abs(abs_coef['Coefficient'])

display(abs_coef.sort_values('Absolute_Coefficient', ascending=False))

print('Interpretation:')
print('- Smoker status usually has the strongest positive effect.')
print('- Age and BMI also significantly increase charges.')
print('- Children and region generally have smaller effects.')


In [ ]:
# Question 15: Compare charges between smokers and non-smokers. How does smoking status affect predictions?

comparison = df.groupby('smoker')['charges'].agg(['mean','median','max'])
display(comparison)

print('Smokers generally have much higher medical charges than non-smokers.')
print('The regression model therefore predicts substantially higher charges for smokers.')


In [ ]:
# Question 16: How would you handle a new client whose region is not in the dataset?

print('Answer:')
print('Use One-Hot Encoding with handle_unknown="ignore" in production pipelines.')
print('The unseen region can be encoded as all zeros for region-related columns or grouped into an "Other" category.')


In [ ]:
# Question 17: What are the limitations of using Multiple Linear Regression for this problem?

print('1. Assumes linear relationships between features and charges.')
print('2. Sensitive to outliers.')
print('3. Cannot automatically capture complex feature interactions.')
print('4. May underperform compared to Random Forest, XGBoost, or Gradient Boosting.')
print('5. Assumes independent features and homoscedastic errors.')
